In [ ]:
# 20 — Live stream: Lakebase CDF → silver → scorecards → Lakebase, in under a minute
# Lakebase CDF appends every change of maintops.incidents to lb_incidents_history (~15 s). This notebook
# streams that table and, per micro-batch:
#   1. MERGEs the latest version of each changed incident into silver (violations → quarantine),
#   2. recomputes scorecards + review sentiment for the affected handymen only, and upserts them to Lakebase,
#   3. records end-to-end latency (Postgres commit → scorecard in Lakebase) in latency_metrics.
# Bursts are safe: the checkpoint guarantees each change is processed exactly once, and anything arriving
# during a micro-batch is picked up by the next one. LLM review summaries are kept off this fast path:
# a sweep every minute regenerates them (at most every SUMMARY_REFRESH_MINUTES per handyman).
# Runs until cancelled (max_minutes = 0) or for max_minutes.

In [ ]:
%pip install -q "psycopg[binary]>=3.1"

In [ ]:
try:
    dbutils.library.restartPython()
except NameError:
    pass

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")
pl.ensure_tables(spark)

import cloudpickle
cloudpickle.register_pickle_by_value(pl)   # ship pipeline_lib with the foreachBatch function

PG_URL = dbutils.secrets.get("maintops", "lakebase_pg_url")
dbutils.widgets.text("poll_seconds", "5")
dbutils.widgets.text("max_minutes", "0")
POLL = int(dbutils.widgets.get("poll_seconds"))
MAX_MINUTES = int(dbutils.widgets.get("max_minutes"))
CHECKPOINT = f"{pl.CHECKPOINT_DIR}/live"

pl.ensure_feedback_table(spark)
spark.sql(f"""CREATE TABLE IF NOT EXISTS {pl.T_LATENCY} (
                batch_id BIGINT, processed_at TIMESTAMP, n_changes BIGINT, n_handymen BIGINT,
                p50_s DOUBLE, p95_s DOUBLE, max_s DOUBLE)
              COMMENT 'End-to-end latency per micro-batch: Postgres commit (_timestamp) → scorecards in Lakebase'""")

In [ ]:
def push_scorecards(s, ids):
    """Fast path: recompute these handymen's scorecards and upsert them to Lakebase straight from the
    driver, before anything is written to Delta. Returns the DataFrame for the gold MERGE afterwards."""
    perf = pl.compute_performance(s.table(pl.T_SILVER).where(F.col("handyman_user_id").isin(ids)))
    rows = perf.collect()                                  # a few rows per affected handyman
    pl.upsert_lakebase(PG_URL, rows, "handyman_performance",
                       ["handyman_user_id", "incident_type"], pl.PERFORMANCE_COLUMNS)
    return s.createDataFrame(rows, schema=perf.schema)


def refresh_handymen(s, ids, summaries=False, perf=None):
    """Persist scorecards to gold and recompute feedback for these handymen (Delta + Lakebase).
    summaries=False never calls the LLM; the sweep passes True to regenerate summaries."""
    affected = s.table(pl.T_SILVER).where(F.col("handyman_user_id").isin(ids))
    if perf is None:
        perf = push_scorecards(s, ids)
    pl.merge_performance(s, perf)

    fb = pl.refresh_summaries(s, pl.compute_feedback(s, affected), throttle=True, generate=summaries)
    pl.merge_feedback(s, fb, stage_table="_stage_feedback_live")
    pl.upsert_lakebase(PG_URL, s.table(pl.fq("_stage_feedback_live")), "handyman_feedback",
                       ["handyman_user_id"], pl.FEEDBACK_COLUMNS)


def process(batch, batch_id):
    s = batch.sparkSession
    changes = batch.where("_pg_change_type IN ('insert', 'update_postimage', 'delete')")
    if changes.isEmpty():
        return

    # Latest change per incident in this micro-batch (ordered by Postgres log position)
    latest = pl.latest_per_id(changes, ["_pg_lsn", "_sort_by"])
    valid, rejected = pl.clean_incidents(latest.where("_pg_change_type <> 'delete'"), "lakebase_cdf")
    rejected.write.mode("append").saveAsTable(pl.T_QUARANTINE)
    pl.merge_into_silver(s, valid)
    latest.where("_pg_change_type = 'delete'").select("id").createOrReplaceTempView("_deleted")
    s.sql(f"MERGE INTO {pl.T_SILVER} t USING _deleted d ON t.id = d.id WHEN MATCHED THEN DELETE")

    # Every handyman touched by a change (old and new assignee) gets a fresh scorecard
    ids = [r[0] for r in batch.where("handyman_user_id IS NOT NULL")
                              .select("handyman_user_id").distinct().collect()]
    perf = push_scorecards(s, ids) if ids else None

    # Latency = Postgres commit → scorecards live in Lakebase (measured before the slower Delta bookkeeping)
    done_at = time.time()
    lat = changes.select((F.lit(done_at) - F.unix_timestamp(F.col("_timestamp").cast("timestamp"))).alias("s"))
    q = lat.agg(F.count("*").alias("n"), F.percentile_approx("s", 0.5).alias("p50"),
                F.percentile_approx("s", 0.95).alias("p95"), F.max("s").alias("max")).first()
    if ids:
        refresh_handymen(s, ids, perf=perf)
    s.createDataFrame([(batch_id, q.n, len(ids), float(q.p50), float(q.p95), float(q["max"]))],
                      "batch_id BIGINT, n_changes BIGINT, n_handymen BIGINT, p50_s DOUBLE, p95_s DOUBLE, max_s DOUBLE") \
     .withColumn("processed_at", F.current_timestamp()) \
     .select("batch_id", "processed_at", "n_changes", "n_handymen", "p50_s", "p95_s", "max_s") \
     .write.mode("append").saveAsTable(pl.T_LATENCY)


def sweep_held_back_summaries():
    """Regenerate summaries off the fast path: handymen with reviews newer than their summary
    (at most once per SUMMARY_REFRESH_MINUTES each) or who just reached enough reviews for one."""
    stale = [r[0] for r in spark.table(pl.T_GOLD_FEEDBACK)
             .where(f"recent_review_count >= {pl.SUMMARY_MIN_REVIEWS} AND ("
                    f"review_summary IS NULL OR (last_review_at > summary_review_at AND "
                    f"summary_updated_at < current_timestamp() - INTERVAL {pl.SUMMARY_REFRESH_MINUTES} MINUTES))")
             .select("handyman_user_id").limit(500).collect()]
    if stale:
        refresh_handymen(spark, stale, summaries=True)
        print(f"sweep: refreshed summaries for {len(stale)} handymen")

In [ ]:
def start(trigger):
    # skipChangeCommits: a schema migration makes Lakebase CDF re-snapshot (rewrite) the history table;
    # skip that rewrite and keep streaming the appended changes instead of failing
    return (spark.readStream.option("skipChangeCommits", "true").table(pl.T_LB_INCIDENTS)
            .writeStream.foreachBatch(process)
            .option("checkpointLocation", CHECKPOINT)
            .trigger(**trigger)
            .start())

deadline = time.time() + MAX_MINUTES * 60 if MAX_MINUTES else None
last_sweep = time.time()
try:
    query = start({"processingTime": f"{POLL} seconds"})     # one long-running query
    print(f"Streaming with a {POLL}s trigger")
    while query.isActive and (deadline is None or time.time() < deadline):
        time.sleep(30)
        if time.time() - last_sweep > 60:
            sweep_held_back_summaries()
            last_sweep = time.time()
    query.stop()
except Exception as e:
    if "processingtime" not in str(e).lower() and "trigger" not in str(e).lower():
        raise
    # Serverless may only allow availableNow: run it back-to-back instead (same checkpoint semantics)
    print(f"processingTime trigger unavailable ({e.__class__.__name__}); polling every {POLL}s with availableNow")
    while deadline is None or time.time() < deadline:
        q = start({"availableNow": True})
        q.awaitTermination()
        n = pl.rows_processed(q)
        if n:
            print(f"{time.strftime('%H:%M:%S')} processed {n} changes")
        if time.time() - last_sweep > 60:
            sweep_held_back_summaries()
            last_sweep = time.time()
        time.sleep(POLL)